In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler


In [3]:
df = pd.read_csv("raw_server_responses.csv")
df.head()


,url,method,param_name,input_type,default_value,form_action,input_value,label,response_status,response_time,html_content_length,error_message_flag,response_body_path
0,http://localhost:8080/login.php,POST,username,text,NaN,login.php,test,safe,404,6.070375,0,0,outputs/response_logs/response_0.html
1,http://localhost:8080/login.php,POST,username,text,NaN,login.php,' OR '1'='1' --,injection,404,4.441500,0,0,outputs/response_logs/response_1.html
2,http://localhost:8080/login.php,POST,username,text,NaN,login.php,1; DROP TABLE users --,injection,404,2.960443,0,0,outputs/response_logs/response_2.html
3,http://localhost:8080/login.php,POST,password,password,NaN,login.php,test,safe,404,4.277945,0,0,outputs/response_logs/response_3.html
4,http://localhost:8080/login.php,POST,password,password,NaN,login.php,' OR '1'='1' --,injection,404,2.111673,0,0,outputs/response_logs/response_4.html


In [4]:
df.drop_duplicates(inplace=True)
df.dropna(subset=['response_status', 'response_time', 'html_content_length'], inplace=True)

# Optional: Convert inconsistent labels
df['label'] = df['label'].replace({'safe': 0, 'injection': 1})


/tmp/ipython-input-4269688896.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['label'] = df['label'].replace({'safe': 0, 'injection': 1})


In [5]:
# Map response codes to status groups
def map_status(code):
    if 200 <= code < 300:
        return '2xx'
    elif 400 <= code < 500:
        return '4xx'
    elif 500 <= code < 600:
        return '5xx'
    else:
        return 'other'

df['status_group'] = df['response_status'].apply(map_status)

# Create the is_malicious flag
df['is_malicious'] = df['label']

# Keep only the relevant columns
features = df[['url', 'response_time', 'html_content_length', 'error_message_flag', 'status_group', 'is_malicious']]


In [6]:
# Normalized response time
features['response_time_norm'] = features.groupby('url')['response_time'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))

# Content length delta (difference between safe and malicious for same URL)
safe_mean = features[features['is_malicious'] == 0].groupby('url')['html_content_length'].mean()
mal_mean = features[features['is_malicious'] == 1].groupby('url')['html_content_length'].mean()
delta = (mal_mean - safe_mean).fillna(0)
features['content_length_delta'] = features['url'].map(delta)


/tmp/ipython-input-4046184120.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features['response_time_norm'] = features.groupby('url')['response_time'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
/tmp/ipython-input-4046184120.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features['content_length_delta'] = features['url'].map(delta)


In [7]:
scaler = MinMaxScaler()
features[['response_time', 'html_content_length']] = scaler.fit_transform(
    features[['response_time', 'html_content_length']]
)

/tmp/ipython-input-830603824.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  features[['response_time', 'html_content_length']] = scaler.fit_transform(


In [8]:
features = pd.get_dummies(features, columns=['status_group'], drop_first=True)


In [10]:
features.to_csv("feature_dataset.csv", index=False)
print("✅ Feature dataset saved at feature_dataset.csv")


✅ Feature dataset saved at feature_dataset.csv
